# Rullende hul cylinder: vinkelhastighed, vinkelacceleration og kinetisk energi

Notebooken samler dine data og regner med usikkerheder.
Alle formler står i almindelig tekst (ingen LaTeX).

## 1) Installer pakker (kør kun hvis nødvendigt)
Hvis du mangler pakker, så kør cellen herunder én gang.

In [ ]:
# Kør kun hvis du mangler pakkerne
# %pip install numpy uncertainties matplotlib

## 2) Importer pakker

In [ ]:
import csv
from pathlib import Path
import numpy as np
from uncertainties import ufloat

## 3) Inddata med usikkerheder

Her er de værdier du gav:
- masse m = 365.14 g
- ydre radius mod rampen R = 5.0 cm
- indre radius r = 1.5 cm
- rampelængde s = 83 cm
- rampehøjde h = 18 cm
- ren rulning uden slip

Usikkerhederne nedenfor kan du justere, hvis du har bedre måleusikkerheder.

In [ ]:
# SI-enheder
m = ufloat(0.36514, 0.00001)   # kg  (0.01 g)
R = ufloat(0.0500, 0.0005)     # m   (0.05 cm)
r = ufloat(0.0150, 0.0005)     # m   (0.05 cm)
s = ufloat(0.830, 0.005)       # m   (0.5 cm)
h = ufloat(0.180, 0.005)       # m   (0.5 cm)
g = ufloat(9.82, 0.01)         # m/s^2

## 4) Teori: ligninger

Brugte ligninger:
- træghedsmoment for hul cylinder: I = 0.5 * m * (R^2 + r^2)
- sin(theta) = h / s
- acceleration langs rampen: a = g * sin(theta) / (1 + I / (m * R^2))
- vinkelacceleration: alpha = a / R
- sluthastighed fra hvile: v = sqrt(2 * a * s)
- vinkelhastighed: omega = v / R
- translationsenergi: E_trans = 0.5 * m * v^2
- rotationsenergi: E_rot = 0.5 * I * omega^2
- total kinetisk energi: E_kin = E_trans + E_rot

In [ ]:
I = 0.5 * m * (R**2 + r**2)
sin_theta = h / s
a = g * sin_theta / (1 + I / (m * R**2))
alpha = a / R
v = (2 * a * s) ** 0.5
omega = v / R
E_trans = 0.5 * m * v**2
E_rot = 0.5 * I * omega**2
E_kin = E_trans + E_rot
E_pot = m * g * h

print('I [kg m^2]      =', I)
print('sin(theta)      =', sin_theta)
print('a [m/s^2]       =', a)
print('alpha [rad/s^2] =', alpha)
print('v [m/s]         =', v)
print('omega [rad/s]   =', omega)
print('E_trans [J]     =', E_trans)
print('E_rot [J]       =', E_rot)
print('E_kin [J]       =', E_kin)
print('mgh [J]         =', E_pot)

## 5) Læs gyrometer-data fra de 5 forsøg

For hvert run estimeres:
- omega_start = gennemsnit af de første 10% punkter
- omega_end = gennemsnit af de sidste 10% punkter
- alpha_fit = hældning fra lineær fit af omega(t)
- alpha_dt = (omega_end - omega_start) / delta_t

In [ ]:
def load_gyro_file(path):
    t = []
    w = []
    with open(path, newline='', encoding='utf-8-sig') as f:
        reader = csv.DictReader(f)
        for row in reader:
            t.append(float(row['time_s']))
            w.append(float(row['gyro_y_rad_s']))
    return np.array(t), np.array(w)

base = Path('.')
files = [base / f'gyro_run_{i}.csv' for i in range(1, 6)]

rows = []
for file in files:
    t, w = load_gyro_file(file)
    n = len(w)
    k = max(3, int(0.1 * n))

    omega_start = float(np.mean(w[:k]))
    omega_end = float(np.mean(w[-k:]))
    dt = float(t[-1] - t[0])

    slope, intercept = np.polyfit(t, w, 1)
    alpha_fit = float(slope)
    alpha_dt = (omega_end - omega_start) / dt

    rows.append({
        'run': file.name,
        'points': n,
        'omega_start': omega_start,
        'omega_end': omega_end,
        'delta_t': dt,
        'alpha_fit': alpha_fit,
        'alpha_dt': alpha_dt,
    })

for rdata in rows:
    print(rdata)

## 6) Saml eksperimentelle estimater med usikkerhed

Her bruges middelværdi +/- standardafvigelse mellem runs.

In [ ]:
alpha_fit_vals = np.array([r['alpha_fit'] for r in rows], dtype=float)
alpha_dt_vals = np.array([r['alpha_dt'] for r in rows], dtype=float)
omega_end_vals = np.array([r['omega_end'] for r in rows], dtype=float)

# sample standard deviation (ddof=1)
def mean_std(x):
    mu = float(np.mean(x))
    if len(x) > 1:
        sd = float(np.std(x, ddof=1))
    else:
        sd = 0.0
    return mu, sd

mu_alpha_fit, sd_alpha_fit = mean_std(alpha_fit_vals)
mu_alpha_dt, sd_alpha_dt = mean_std(alpha_dt_vals)
mu_omega_end, sd_omega_end = mean_std(omega_end_vals)

alpha_fit_u = ufloat(mu_alpha_fit, sd_alpha_fit)
alpha_dt_u = ufloat(mu_alpha_dt, sd_alpha_dt)
omega_end_u = ufloat(mu_omega_end, sd_omega_end)

v_end_u = R * omega_end_u
E_trans_meas = 0.5 * m * v_end_u**2
E_rot_meas = 0.5 * I * omega_end_u**2
E_kin_meas = E_trans_meas + E_rot_meas

print('alpha_fit [rad/s^2] =', alpha_fit_u)
print('alpha_dt  [rad/s^2] =', alpha_dt_u)
print('omega_end [rad/s]   =', omega_end_u)
print('v_end [m/s]         =', v_end_u)
print('E_kin,meas [J]      =', E_kin_meas)

## 7) Sammenligning teori vs målt

In [ ]:
print('TEORI')
print('alpha_theory [rad/s^2] =', alpha)
print('omega_theory [rad/s]   =', omega)
print('E_kin_theory [J]       =', E_kin)
print()
print('MÅLT (fra gyro-filer)')
print('alpha_fit [rad/s^2]    =', alpha_fit_u)
print('alpha_dt [rad/s^2]     =', alpha_dt_u)
print('omega_end [rad/s]      =', omega_end_u)
print('E_kin_meas [J]         =', E_kin_meas)

## 8) Samlet plot for alle 5 runs

Dette plot viser gyro_y_rad_s som funktion af tid for alle runs i samme figur.


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
for i in range(1, 6):
    file = Path(f'gyro_run_{i}.csv')
    t, w = load_gyro_file(file)
    t_rel = t - t[0]
    plt.plot(t_rel, w, label=f'Run {i}', linewidth=1.2)

plt.title('Gyro vinkelhastighed for 5 runs')
plt.xlabel('Tid siden start [s]')
plt.ylabel('Vinkelhastighed omega [rad/s]')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()
